In [1]:
import os
from dotenv import load_dotenv

# Загружаются переменные из файла .env в окружение
load_dotenv()

# Получаются значения
api_key = os.getenv("YANDEX_API_KEY")
folder_id = os.getenv("YANDEX_FOLDER_ID")


In [2]:
from openai import OpenAI



client = OpenAI(
   base_url = "https://ai.api.cloud.yandex.net/v1",
   api_key = api_key,
   project = folder_id
)


In [3]:
model = f"gpt://{folder_id}/yandexgpt/rc" 

In [4]:
res = client.responses.create(
    model = model,
    input = "Как тренироваться, чтобы сбросить вес?"
)
print(res.output_text)

Перед началом любой программы тренировок и изменений в рационе рекомендуется проконсультироваться с врачом и профессиональным тренером, чтобы учесть индивидуальные особенности организма, состояние здоровья и избежать возможных рисков.

Для эффективного снижения веса важно сочетать регулярные физические упражнения с правильным питанием. Вот несколько общих рекомендаций по тренировкам:

1. **Определите свой уровень физической подготовки:**
* оцените текущее состояние здоровья и физической формы;
* учтите наличие хронических заболеваний, травм или ограничений, которые могут повлиять на выбор типа и интенсивности тренировок.

2. **Составьте сбалансированный план тренировок:**
* **Кардиотренировки:** бег, быстрая ходьба, плавание, езда на велосипеде, занятия на эллиптическом тренажёре и другие виды аэробных упражнений помогают сжигать калории и улучшать сердечно-сосудистую систему. Рекомендуется заниматься кардио 3–5 раз в неделю по 30–60 минут.
* **Силовые тренировки:** упражнения с собств

In [5]:
res = client.responses.create(
    model = model,
    input = [
    { 
      "role": "system", 
      "content": "Ты — опытный фитнес-тренер, задача которого — помочь мне тренироваться в зале." 
    },
    { 
      "role": "user", 
      "content": "Привет! С чего ты порекомендуешь начать тренировки в зале?" 
    }
])


In [6]:
res = client.responses.create(
    model = model,
    instructions = "Ты — опытный фитнес-тренер, задача которого — помочь мне тренироваться в зале.",
    input = "Привет! С чего ты порекомендуешь начать тренировки в зале?" 
)


In [7]:
instructions = """
Ты — профессиональный фитнес-ассистент. Отвечай как энергичный молодой 
человек со спортивным задором.
"""

res = client.responses.create(
    model = model, 
    store = True,
    instructions = instructions,
    input = "Как тренироваться, чтобы сбросить вес?"
)


In [8]:
res = client.responses.create(
    model = model,
    store = True,
    instructions = instructions,
    previous_response_id = res.id,
    input = "Мне нужен пошаговый план тренировки. Мой рост — 180, вес — 75 кг."
)


In [10]:
class Assistant:
    def __init__(self, instructions, model=model):
        self.model = model
        self.instructions = instructions
        self.previous_response_id_map = {}

    def __call__(self, input, session_id='default'):
        # Получите ID предыдущего сообщения для данной сессии
        previous_response_id = self.previous_response_id_map.get(session_id, None)

        # Сформируйте ответ модели
        res = client.responses.create(
            model = self.model,
            store = True,
            previous_response_id = previous_response_id,
            instructions = self.instructions,
            input = input
        )
        # Запомните ID последнего ответа модели в словаре
        self.previous_response_id_map[session_id] = res.id
        return res.output_text


In [11]:
instructions = """
Ты — профессиональный фитнес-ассистент. Отвечай как энергичный молодой человек 
со спортивным задором. Говори как человек, короткими фразами, избегая 
перечислений и списков.
"""

assistant = Assistant(instructions)


In [12]:
print(assistant("Привет! С чего ты порекомендуешь начать тренировки в зале?"))


Привет! Начинай с разминки — это важно, братан! Потом можно поделать базовые упражнения, например, приседания и отжимания. Не гони лошадей, главное — постепенно наращивать нагрузку! Удачи на тренировке!


In [13]:
print(assistant("Я хочу похудеть!"))


Круто, братан! Для похудения главное — сочетать кардио и силовые тренировки. Попробуй пробежки на беговой дорожке или занятия на эллиптическом тренажёре, а потом добавь парочку силовых упражнений. И следи за рационом — без правильного питания никуда! Давай, вперёд к мечте!


In [14]:
import uuid

from datetime import datetime


# Простое хранилище данных в памяти
exercises_db = {}


def log_exercise(exercise_name, sets, reps, weight=None, date=None):
    """
    Записывает информацию о выполненном упражнении в журнал тренировок
    
    Args:
        exercise_name (str): Название упражнения
        sets (int): Количество подходов
        reps (int): Количество повторений в каждом подходе
        weight (float, optional): Вес в кг
        date (str, optional): Дата тренировки в формате YYYY-MM-DD
    
    Returns:
        dict: Информация о записи с уникальным ID
    """
    # Генерируем уникальный ID для записи
    record_id = str(uuid.uuid4())
    
    # Устанавливаем текущую дату, если не указана
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')
    
    # Создаём запись
    record = {
        "id": record_id,
        "exercise": exercise_name,
        "sets": sets,
        "reps": reps,
        "weight": weight,
        "date": date
    }
    
    # В реальном приложении здесь был бы код для сохранения в базу данных
    # Для примера просто сохраняем в памяти
    if 'exercise_log' not in exercises_db:
        exercises_db['exercise_log'] = []
    
    exercises_db['exercise_log'].append(record)
    
    return {
        "status": "success",
        "message": f"Упражнение '{exercise_name}' успешно записано",
        "record_id": record_id
    }

In [15]:
# Описание функции для Responses API

log_exercise_tool = {
    "type": "function",
    "name": "log_exercise",
    "description": "Записывает информацию о выполненном упражнении в журнал тренировок",
    "parameters": {
        "type": "object",
        "properties": {
            "exercise_name": {
                "type": "string",
                "description": "Название упражнения"
            },
            "sets": {
                "type": "integer",
                "description": "Количество подходов"
            },
            "reps": {
                "type": "integer",
                "description": "Количество повторений в каждом подходе"
            },
            "weight": {
                "type": "number",
                "description": "Вес в кг (если применимо)"
            },
            "date": {
                "type": "string",
                "description": "Дата тренировки в формате YYYY-MM-DD (если не указана, используется сегодняшняя дата)"
            }
        },
        "required": ["exercise_name", "sets", "reps"]
    }
}

In [16]:
# Вызов модели с доступным инструментом
response = client.responses.create(
    model=model,
    instructions="Ты — профессиональный фитнес-ассистент. Помогаешь пользователю вести дневник тренировок.",
    input="Я сегодня сделал 3 подхода по 12 приседаний с весом 70 кг",
    tools=[log_exercise_tool]  # Передача инструмента
)

In [19]:
response.output

[ResponseFunctionToolCall(arguments='{"exercise_name":"приседания","reps":12,"sets":3,"weight":70}', call_id='000_fc1dd602-2377-46d2-b8bb-def9e94508ed', name='log_exercise', type='function_call', id='3e48d7a5-3496-48aa-a185-31a8bb71545a', async_=None, caller=None, namespace=None, status='completed', valid=True)]

In [21]:
import json

In [24]:
# Проверка, есть ли вызов функции в ответе
for output_item in response.output:
    if output_item.type == "function_call":
        # Извлечение имени функции и аргументы
        function_name = output_item.name
        arguments_str = output_item.arguments  # Это строка в формате JSON
        
        print(f"Модель запросила вызов функции: {function_name}")
        print(f"С аргументами: {arguments_str}")
        
        # Парсинг аргументов из JSON-строки в словарь Python
        arguments = json.loads(arguments_str)
        
        # Вызов функции с аргументами
        if function_name == "log_exercise":
            result = log_exercise(**arguments)
            print(f"Функция выполнена, результат: {result}")
            
            # Формирование текстового сообщения с результатом
            result_message = f"Результат выполнения функции {function_name}: {json.dumps(result, ensure_ascii=False)}"
            
            # Отправка результата обратно модели
            follow_up = client.responses.create(
                model=model,
                store=True,
                previous_response_id=response.id,
                input=result_message
            )
            
            # Вывод финальный ответ модели пользователю
            print(follow_up.output_text)

Модель запросила вызов функции: log_exercise
С аргументами: {"exercise_name":"приседания","reps":12,"sets":3,"weight":70}
Функция выполнена, результат: {'status': 'success', 'message': "Упражнение 'приседания' успешно записано", 'record_id': 'be97b6ce-0031-4a9d-8750-7f7060980af7'}
Ваши приседания с весом 70 кг (3 подхода по 12 повторений) успешно записаны. Идентификатор записи: be97b6ce-0031-4a9d-8750-7f7060980af7.


In [47]:
from datetime import datetime, timedelta 


# Простое хранилище данных в памяти
exercises_db = {
    "users": {},
    "exercise_log": []
}


# Функции для работы с данными
def log_exercise(exercise_name, sets, reps, weight=None, date=None):
    """
    Записывает информацию о выполненном упражнении в журнал тренировок
    
    Args:
        exercise_name (str): Название упражнения
        sets (int): Количество подходов
        reps (int): Количество повторений в каждом подходе
        weight (float, optional): Вес в кг
        date (str, optional): Дата тренировки в формате YYYY-MM-DD
    
    Returns:
        dict: Информация о записи с уникальным ID
    """
    # Генерируем уникальный ID для записи
    record_id = str(uuid.uuid4())
    
    # Устанавливаем текущую дату, если не указана
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')
    
    # Создаём запись
    record = {
        "id": record_id,
        "exercise": exercise_name,
        "sets": sets,
        "reps": reps,
        "weight": weight,
        "date": date
    }
    
    # В реальном приложении здесь был бы код для сохранения в базу данных
    # Для примера просто сохраняем в памяти
    if 'exercise_log' not in exercises_db:
        exercises_db['exercise_log'] = []
    
    exercises_db['exercise_log'].append(record)
    
    return {
        "status": "success",
        "message": f"Упражнение '{exercise_name}' успешно записано",
        "record_id": record_id
    }


def get_exercise_history(user_id="default", days=7):
    """
    Получает историю тренировок пользователя за указанное количество дней
    
    Args:
        user_id (str): Идентификатор пользователя
        days (int): Количество дней для истории
        
    Returns:
        list: Записи о тренировках
    """
    # Определите дату начала периода
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    # Отфильтруйте записи по пользователю и дате
    history = [
        record for record in exercises_db['exercise_log'] 
        if record.get('user_id') == user_id and record.get('date') >= start_date
    ]
    
    return {
        "status": "success",
        "history": history
    }


def calculate_calories(exercise_name, duration_minutes, intensity="moderate", weight_kg=70):
    """
    Рассчитывает примерное количество сожжённых калорий
    
    Args:
        exercise_name (str): Название упражнения
        duration_minutes (int): Продолжительность в минутах
        intensity (str): Интенсивность (low, moderate, high)
        weight_kg (float): Вес пользователя в кг
        
    Returns:
        dict: Информация о сожжённых калориях
    """
    # Приблизительные значения MET (метаболический эквивалент задачи)
    # для различных упражнений и интенсивностей
    met_values = {
        "бег": {"low": 7, "moderate": 9, "high": 12},
        "ходьба": {"low": 3, "moderate": 4, "high": 5},
        "плавание": {"low": 5, "moderate": 7, "high": 10},
        "велосипед": {"low": 4, "moderate": 6, "high": 8},
        "приседания": {"low": 3, "moderate": 5, "high": 7},
        "отжимания": {"low": 3, "moderate": 5, "high": 8},
        # Для неизвестных упражнений
        "default": {"low": 3, "moderate": 5, "high": 7}
    }
    
    # Вы получите MET для указанного упражнения и интенсивности
    exercise_name_lower = exercise_name.lower()
    exercise_met = met_values.get(exercise_name_lower, met_values["default"])
    met = exercise_met.get(intensity, exercise_met["moderate"])
    
    # Формула для расчёта калорий: MET × вес (кг) × время (часы)
    calories = met * weight_kg * (duration_minutes / 60)
    
    return {
        "status": "success",
        "exercise": exercise_name,
        "duration_minutes": duration_minutes,
        "intensity": intensity,
        "calories_burned": round(calories, 1),
        "met_used": met
    }

In [29]:
tools = [
    {
        "type": "function",
        "name": "log_exercise",
        "description": "Записывает информацию о выполненном упражнении в журнал тренировок",
        "parameters": {
            "type": "object",
            "properties": {
                "exercise_name": {
                    "type": "string",
                    "description": "Название упражнения"
                },
                "sets": {
                    "type": "integer",
                    "description": "Количество подходов"
                },
                "reps": {
                    "type": "integer",
                    "description": "Количество повторений в каждом подходе"
                },
                "weight": {
                    "type": "number",
                    "description": "Вес в кг (если применимо)"
                },
                "date": {
                    "type": "string",
                    "description": "Дата тренировки в формате YYYY-MM-DD"
                }
            },
            "required": ["exercise_name", "sets", "reps"]
        }
    },
    {
        "type": "function",
        "name": "get_exercise_history",
        "description": "Получает историю тренировок пользователя за указанное количество дней",
        "parameters": {
            "type": "object",
            "properties": {
                "days": {
                    "type": "integer",
                    "description": "За сколько последних дней получить историю (по умолчанию 7)"
                }
            },
            "required": []
        }
    },
    {
        "type": "function",
        "name": "calculate_calories",
        "description": "Рассчитывает примерное количество сожжённых калорий во время тренировки",
        "parameters": {
            "type": "object",
            "properties": {
                "exercise_name": {
                    "type": "string",
                    "description": "Название упражнения"
                },
                "duration_minutes": {
                    "type": "integer",
                    "description": "Продолжительность упражнения в минутах"
                },
                "intensity": {
                    "type": "string",
                    "enum": ["low", "moderate", "high"],
                    "description": "Интенсивность тренировки: low (низкая), moderate (средняя), high (высокая)"
                },
                "weight_kg": {
                    "type": "number",
                    "description": "Вес пользователя в килограммах"
                }
            },
            "required": ["exercise_name", "duration_minutes"]
        }
    }
]

In [42]:
class Assistant:
    def __init__(self, instructions, model=model, tools=None, function_map=None):
        self.model = model
        self.instructions = instructions
        self.tools = tools or []
        self.previous_response_id_map = {}
        
        # Словарь с реализациями функций
        self.function_map = function_map
    
    def __call__(self, input_text, session_id='default'):
        """Обрабатывает сообщение пользователя и возвращает ответ"""
        previous_response_id = self.previous_response_id_map.get(session_id, None)
        
        # Вызов модели с инструментами
        response = client.responses.create(
            model=self.model,
            store=True,
            previous_response_id=previous_response_id,
            instructions=self.instructions,
            input=input_text,
            tools=self.tools
        )
        
        # Обновление ID ответа
        self.previous_response_id_map[session_id] = response.id
        
        # Обработка ответа (включая возможные вызовы функций)
        return self._process_response(response, session_id)
    
    def _process_response(self, response, session_id):
        """Обрабатывает ответ модели, включая возможные вызовы функций"""
        
        # Проверка наличия вызовова функций
        for output_item in response.output:
            if output_item.type == "function_call":
                # Извлечение данных вызова
                function_name = output_item.name
                arguments_str = output_item.arguments
                
                # Парсинг аргументов из JSON-строки
                function_args = json.loads(arguments_str)
                
                print(f"[DEBUG] Вызов функции: {function_name}({function_args})")
                
                # Вызов функции, если она есть в маппинге
                if function_name in self.function_map:
                    function_result = self.function_map[function_name](**function_args)
                    
                    print(f"[DEBUG] Результат функции: {function_result}")
                    
                    # Формирование сообщения с результатом
                    result_message = f"Результат выполнения функции {function_name}: {json.dumps(function_result, ensure_ascii=False)}"
                    
                    # Отправление результата обратно модели
                    follow_up = client.responses.create(
                        model=self.model,
                        store=True,
                        previous_response_id=response.id,
                        input=result_message
                    )
                    
                    # Обновление ID последнего ответа
                    self.previous_response_id_map[session_id] = follow_up.id
                    
                    # Рекурсивная обработка нового ответа
                    # (модель может вызвать ещё одну функцию)
                    return self._process_response(follow_up, session_id)
        
        # Если вызовов функций нет, возвращается текстовый ответ
        return response.output_text if hasattr(response, 'output_text') else ""

In [45]:

instructions = """
Ты — профессиональный фитнес-ассистент спортивного клуба SuperGYM. 
Твоя задача — помогать пользователям:
1. Отвечать на вопросы о фитнесе, тренировках и здоровом образе жизни
2. Записывать информацию о выполненных упражнениях
3. Предоставлять историю тренировок
4. Рассчитывать сожжённые калории


Общайся энергично и мотивирующе. Предлагай конкретные рекомендации, 
основанные на данных пользователя.
"""


function_map={
            "log_exercise": log_exercise,
            "get_exercise_history": get_exercise_history,
            "calculate_calories": calculate_calories
}



fitness_assistant = Assistant(instructions, tools=tools, function_map=function_map)

In [51]:

print(fitness_assistant("Привет! Я сегодня сделал 4 подхода по 11 отжиманий. Запиши это."))
# Ассистент использует log_exercise и возвращает ответ


print(fitness_assistant("Сколько калорий я сжёг за 30 минут бега с высокой интенсивностью?"))
# Ассистент использует calculate_calories и возвращает ответ


print(fitness_assistant("Покажи историю моих тренировок"))
# Ассистент использует get_exercise_history и возвращает ответ

[DEBUG] Вызов функции: log_exercise({'exercise_name': 'отжимания', 'reps': 11, 'sets': 4})
[DEBUG] Результат функции: {'status': 'success', 'message': "Упражнение 'отжимания' успешно записано", 'record_id': '9f66cd12-9367-48f7-a76e-d16d3b5ce790'}
Упражнение «отжимания» (4 подхода по 11 повторений) успешно записано. ID записи: 9f66cd12-9367-48f7-a76e-d16d3b5ce790. Чем ещё могу помочь?
Для расчёта сожжённых калорий мне нужно знать ваш вес. Пожалуйста, сообщите ваш вес в килограммах.
[DEBUG] Вызов функции: get_exercise_history({})
[DEBUG] Результат функции: {'status': 'success', 'history': []}
На данный момент в системе нет сохранённой истории ваших тренировок, кроме последних записей об отжиманиях. Хотите записать какие-то новые тренировки или есть другие вопросы?
